### 3.5.1. General Nonlinear Programs

$$
\begin{aligned}
\min_{\mathbf{x}\in\mathbb{R}^n}\quad & f(\mathbf{x}) \\
\text{subject to}\quad & g_i(\mathbf{x}) \le 0,\quad i=1,\dots,m,\\
& h_j(\mathbf{x}) = 0,\quad j=1,\dots,p.
\end{aligned}
$$
First-order (KKT) conditions at a regular minimizer $\mathbf{x}^\star$:
$$
\nabla f(\mathbf{x}^\star) + \sum_i \mu_i \nabla g_i(\mathbf{x}^\star) + \sum_j \lambda_j \nabla h_j(\mathbf{x}^\star) = \mathbf 0,
\quad \mu_i \ge 0,\ \mu_i g_i(\mathbf{x}^\star)=0.
$$

**Explanation:**

The general nonlinear program places no convexity assumption on $f$, $g_i$, or $h_j$. Consequently the [KKT conditions](../03_Convex_Optimization/46_karush_kuhn_tucker_conditions.ipynb) are only *necessary* for optimality: they identify stationary points, but such a point may be a local minimizer, a local maximizer, or a saddle point, and there can be many of them. Establishing that a stationary point is a local minimizer requires a second-order condition on the Hessian of the Lagrangian restricted to the tangent space of the active constraints.

**Assumptions:**
- $f, g_i, h_j$ are twice continuously differentiable.
- A constraint qualification holds at $\mathbf{x}^\star$ so multipliers exist.

**Numerical Example:**

The unconstrained function
$$
f(x) = x^4 - 3x^2 + x
$$
is nonconvex. Its stationary points solve $f'(x) = 4x^3 - 6x + 1 = 0$, which has three real roots:
$$
x \approx -1.301,\quad x \approx 0.170,\quad x \approx 1.132.
$$
Evaluating the second derivative $f''(x) = 12x^2 - 6$ classifies them:
$$
f''(-1.301) > 0\ (\text{local min}),\quad
f''(0.170) < 0\ (\text{local max}),\quad
f''(1.132) > 0\ (\text{local min}),
$$
and comparing values shows the *global* minimizer is $x\approx -1.301$. A local solver started near $x=1$ would instead return the inferior local minimizer $x\approx 1.132$.

In [1]:
import sympy as sp

x = sp.symbols("x", real=True)
objective = x**4 - 3*x**2 + x
first_derivative = sp.diff(objective, x)
second_derivative = sp.diff(objective, x, 2)

stationary_points = sorted(float(root) for root in sp.Poly(first_derivative, x).nroots() if root.is_real)
classified = [(round(point, 4),
               "min" if float(second_derivative.subs(x, point)) > 0 else "max",
               round(float(objective.subs(x, point)), 4))
              for point in stationary_points]

print("f'(x) =", first_derivative, "   f''(x) =", second_derivative)
for location, kind, value in classified:
    print(f"x = {location:+.4f}  ({kind})  f = {value:+.4f}")
global_minimizer = min(classified, key=lambda item: item[2])
print("global minimizer:", global_minimizer[0], "value", global_minimizer[2])

f'(x) = 4*x**3 - 6*x + 1    f''(x) = 6*(2*x**2 - 1)
x = -1.3008  (min)  f = -3.5139
x = +0.1699  (max)  f = +0.0841
x = +1.1309  (min)  f = -1.0702
global minimizer: -1.3008 value -3.5139


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x")
objective = decision_variable**4 - 3*decision_variable**2 + decision_variable
solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})

for start in (-1.0, 1.0):
    solution = solver(x0=start)
    print(f"start {start:+.1f} -> local minimizer {float(solution['x']):+.4f}, value {float(solution['f']):+.4f}")

start -1.0 -> local minimizer -1.3008, value -3.5139
start +1.0 -> local minimizer +1.1309, value -1.0702


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.). Athena Scientific.](https://www.athenasc.com/nonlinbook.html)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Solver Interfaces for Convex Models](../04_Convex_Problem_Classes/14_solver_interfaces_for_convex_models.ipynb) | [Next: Equality Constraint Lagrange Multiplier Theorem ➡️](./02_equality_constraint_lagrange_multiplier_theorem.ipynb)